# ENCODE Search

![ENCODE Search](https://proto-bio.github.io/proto-assets/images/tool/encode/hero.png)

This notebook demonstrates `run_encode_search`, which searches the ENCODE portal by facet filters (assay, target, biosample, output type, assembly, file format) and optional free text. It returns processed files or experiments with their accessions, metadata and direct download URLs. Here we find the default IDR thresholded peak file for MAFK TF ChIP-seq in HepG2, then list CTCF ChIP-seq experiments with a free-text search.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("encode")
display_overview("encode")
display_docs_section("encode", "Background")

## Available tools

In [ ]:
display_available_tools("encode")

### `run_encode_search`

`run_encode_search` runs one portal search per query. A query is a mapping of ENCODE facet filters, whose values must match the portal's terms exactly, plus an optional `search_term`. Each hit carries the file or experiment accession, its parent experiment, assay, target, biosample, output type, assembly, size, MD5 checksum, preferred-default flag and, for files, the download URL.

In [ ]:
from proto_tools import (
    EncodeSearchConfig,
    EncodeSearchInput,
    run_encode_search,
)

In [ ]:
# Display docs
display_api_reference("encode", "input", "run_encode_search")

# Default GRCh38 IDR thresholded peaks for MAFK TF ChIP-seq in HepG2
inputs = EncodeSearchInput(
    queries={
        "assay_title": "TF ChIP-seq",
        "target.label": "MAFK",
        "biosample_ontology.term_name": "HepG2",
        "output_type": "IDR thresholded peaks",
        "assembly": "GRCh38",
        "file_format": "bed",
    }
)

In [ ]:
# Display config docs
display_api_reference("encode", "config", "run_encode_search")

# Keep only the portal's preferred default file | see docs above for all fields
config = EncodeSearchConfig(default_only=True)

In [ ]:
# Run the search
output = run_encode_search(inputs, config)

In [ ]:
# Display output docs
display_api_reference("encode", "output", "run_encode_search")

result = output.results[0]
print(f"{result.total_count} matching file(s)")
for hit in result.hits:
    print(f"{hit.accession} from {hit.dataset_accession}: {hit.output_type}, {hit.assembly}, {hit.file_type}")
    print(f"  replicates {hit.biological_replicates}, {hit.file_size} bytes, md5 {hit.md5sum}")
    print(f"  {hit.download_url}")

#### Searching Experiments with Free Text

With `item_type="Experiment"` the tool searches experiments instead of files. Free text in `search_term` is useful when the exact facet terms are unknown, and can be combined with filters. Here we list released CTCF TF ChIP-seq experiments in K562.

In [ ]:
exp_output = run_encode_search(
    EncodeSearchInput(queries={"search_term": "CTCF K562", "filters": {"assay_title": "TF ChIP-seq"}}),
    EncodeSearchConfig(item_type="Experiment", max_results=5),
)

exp_result = exp_output.results[0]
print(f"{exp_result.total_count} matching experiment(s); top {len(exp_result.hits)}:")
for hit in exp_result.hits:
    print(f"{hit.accession} | {hit.target} | {hit.biosample} | {hit.lab} | released {hit.date_released}")

#### Matching Any of Several Values

A filter given as a list matches any of its values, so one search can cover several targets or biosamples. Here we collect the default CTCF peaks for three cell lines in a single request.

In [ ]:
multi_output = run_encode_search(
    EncodeSearchInput(
        queries={
            "assay_title": "TF ChIP-seq",
            "target.label": "CTCF",
            "biosample_ontology.term_name": ["K562", "HepG2", "GM12878"],
            "output_type": "IDR thresholded peaks",
            "assembly": "GRCh38",
        }
    ),
    EncodeSearchConfig(default_only=True, max_results=50),
)

from collections import Counter

multi_result = multi_output.results[0]
print(f"{multi_result.total_count} default CTCF peak files")
print(dict(Counter(hit.biosample for hit in multi_result.hits)))

#### Downloading Files

The search returns metadata only. `output.download(directory)` fetches the file hits from their download URLs, checks each against its ENCODE MD5, and returns the local paths. Files already present with the right MD5 are not downloaded again. It refuses up front when the files add up to more than `max_total_bytes` (1 GB by default): peak and quantification files are a few MB, but signal tracks, alignments and reads are often hundreds of MB to several GB. Pass `accessions` to download only some hits.

In [ ]:
import gzip
from tempfile import TemporaryDirectory

with TemporaryDirectory() as download_dir:
    paths = output.download(download_dir)
    for accession, path in paths.items():
        print(f"{accession}: {path.name}, {path.stat().st_size:,} bytes")
        with gzip.open(path, "rt") as peaks:
            for line in list(peaks)[:3]:
                print("  ", line.rstrip())

## Export Results

Results can be saved as JSON (the full output) or CSV (one row per hit) for downstream use.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

output.export("encode_mafk_hepg2_idr_peaks", export_path=output_dir, file_format="csv")
print(f"Exported to {output_dir / 'encode_mafk_hepg2_idr_peaks.csv'}")